# AURORA-VISION — Multi-Modal Video Understanding Demo

This notebook walks through the end-to-end **AURORA-VISION** pipeline:
1. Load and decode a video
2. Extract CLIP ViT-L/14 visual embeddings
3. Transcribe audio with Whisper-large-v3
4. Perform TrOCR on-screen text detection
5. Fuse all modalities with the CrossModalTransformer
6. Generate a natural-language answer via the LangGraph QA pipeline
7. Visualize attention weights and modality gate scores

In [ ]:
import os
import sys

# Ensure repo root is on the path
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

print("AURORA-VISION environment ready.")

## 1. Configuration

In [ ]:
from utils.config_loader import ConfigLoader
from utils.seed import set_seed

set_seed(42)

video_cfg   = ConfigLoader.load("video_config")
model_cfg   = ConfigLoader.load("model_config")
fusion_cfg  = ConfigLoader.load("fusion_config")

print("video_config :", video_cfg)
print("model_config :", model_cfg)
print("fusion_config:", fusion_cfg)

## 2. Download a Sample Video

We use `YouTubeFetcher` to pull a short public video. Set `VIDEO_URL` to any YouTube link.

In [ ]:
from data.fetchers.youtube_fetcher import YouTubeFetcher

VIDEO_URL = "https://www.youtube.com/watch?v=aqz-KE-bpKQ"  # Big Buck Bunny trailer

fetcher = YouTubeFetcher(output_dir="/tmp/aurora_demo")
video_path, metadata = fetcher.fetch(VIDEO_URL)

print(f"Video saved to: {video_path}")
print(f"Metadata: {metadata}")

## 3. Frame Sampling

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from data.processors.frame_sampler import AdaptiveFrameSampler

sampler = AdaptiveFrameSampler()
frames, timestamps = sampler.sample(video_path, max_frames=16)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i, (frame, ts) in enumerate(zip(frames, timestamps)):
    ax = axes[i // 8][i % 8]
    ax.imshow(frame)
    ax.set_title(f"{ts:.1f}s", fontsize=8)
    ax.axis("off")
plt.suptitle("Adaptive Frame Samples", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Sampled {len(frames)} frames at timestamps: {[f'{t:.2f}s' for t in timestamps]}")

## 4. CLIP ViT-L/14 Visual Embeddings

In [ ]:
import torch
from visual.clip_encoder import CLIPFrameEncoder

clip_encoder = CLIPFrameEncoder()

clip_embeddings = clip_encoder.encode_frames(frames)  # (N, 768)
print(f"CLIP embeddings shape: {clip_embeddings.shape}")

# Visualize cosine similarity matrix
sim_matrix = torch.nn.functional.cosine_similarity(
    clip_embeddings.unsqueeze(1), clip_embeddings.unsqueeze(0), dim=-1
).numpy()

plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Cosine Similarity")
plt.title("CLIP Inter-Frame Cosine Similarity Matrix")
plt.xlabel("Frame Index")
plt.ylabel("Frame Index")
plt.tight_layout()
plt.show()

## 5. Whisper-large-v3 Audio Transcription

In [ ]:
from data.processors.audio_extractor import AudioExtractor
from audio.whisper_transcriber import WhisperTranscriber

audio_path = AudioExtractor().extract(video_path)
transcriber = WhisperTranscriber()
result = transcriber.transcribe(audio_path)

print(f"Language: {result.language}")
print(f"Full transcript:\n{result.text[:500]}...")
print(f"\nSegment count: {len(result.segments)}")
for seg in result.segments[:5]:
    print(f"  [{seg.start:.1f}s – {seg.end:.1f}s] {seg.text}")

## 6. Cross-Modal Transformer Fusion

In [ ]:
from fusion.cross_modal_transformer import CrossModalTransformer
import torch

# Dummy audio + text embeddings for demonstration
audio_emb = torch.randn(len(frames), 512)   # (N, 512)
text_emb  = torch.randn(len(frames), 768)   # (N, 768)

fusion_model = CrossModalTransformer()
fusion_model.eval()

with torch.no_grad():
    fused = fusion_model(clip_embeddings, audio_emb, text_emb)

print(f"Fused representation shape : {fused.fused.shape}")
print(f"Modality gate weights       : {fused.gate_weights}")

## 7. Full LangGraph QA Pipeline

In [ ]:
from pipeline.main import build_pipeline

graph = build_pipeline()

query = "What is the main activity shown in this video?"

state = graph.run(
    video_url=VIDEO_URL,
    query=query,
)

print(f"Query : {query}")
print(f"Answer: {state.get('generated_answer', 'N/A')}")
print(f"\nEvaluation scores: {state.get('evaluation_scores', {})}")
print(f"Pipeline trace   : {state.get('pipeline_trace', [])}")